In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV

In [4]:
df = pd.read_csv("data/adult.csv")

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.replace('?', np.nan, inplace=True)

In [ ]:
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="rocket")
plt.show()

In [6]:
df.drop_duplicates(inplace=True)

In [7]:
df["income"] = df["income"].map({"<=50K": 0, ">50K": 1})

In [8]:
df.drop(columns="fnlwgt", inplace=True)

In [9]:
X = df.drop(columns="income")
y = df['income']

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [11]:
num_cols = X_train.select_dtypes(include=['int64']).columns.tolist()
cat_cols = X_train.select_dtypes(include=['str']).columns.tolist()

In [12]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

## Pipeline

In [24]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocesser = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

dt_pipeline = Pipeline([
    ('preprocesser', preprocesser),
    ('model', DecisionTreeClassifier())
])

rf_pipeline = Pipeline([
    ('preprocesser', preprocesser),
    ('model', RandomForestClassifier())
])

xgb_pipeline = Pipeline([
    ('preprocesser', preprocesser),
    ('model', XGBClassifier(device="cuda"))
])

## Evaluating Default Models

In [ ]:
dt_pipeline.fit(X_train, y_train)
rf_pipeline.fit(X_train, y_train)
xgb_pipeline.fit(X_train, y_train)

In [14]:
from sklearn.metrics import roc_auc_score
from sklearn.metrics import classification_report

In [ ]:
# Decision Tree Default Metrics
print("Decision Tree Default Metrics: \n")
print(f"ROC-AUC Score: {roc_auc_score(y_test, dt_pipeline.predict_proba(X_test)[:, 1]):.3f}\n")
print(f"Classification Report:\n {classification_report(y_test, dt_pipeline.predict(X_test))}")

In [ ]:
# Random forset Default Metrics
print("Random Forest Default Metrics: \n")
print(f"ROC-AUC Score: {roc_auc_score(y_test, rf_pipeline.predict_proba(X_test)[:, 1]):.3f}\n")
print(f"Classification Report:\n {classification_report(y_test, rf_pipeline.predict(X_test))}")

In [ ]:
# XGBoost Default Metrics
print("XGBoost Default Metrics: \n")
print(f"ROC-AUC Score: {roc_auc_score(y_test, xgb_pipeline.predict_proba(X_test)[:, 1]):.3f}\n")
print(f"Classification Report:\n {classification_report(y_test, xgb_pipeline.predict(X_test))}")

### Hyperparameter tuning for models
- Decision Trees -> criterion, max_depth, min_samples_split, min_samples_left
- Random Forest  -> criterion, max_depth, min_samples_split, min_samples_left, n_estimators, max_features
- XGBoost        -> n_estimators, max_depth, learning_rate, subsample

In [15]:
dt_param_grid = {
    'model__criterion': ['gini', 'entropy'],
    'model__min_samples_split': [2, 10, 20],
    'model__min_samples_leaf': [1, 5, 10],
    'model__max_depth': [3, 5, 10, None]
}
rf_param_grid = {
    'model__n_estimators': [100, 200, 300],
    'model__max_features': ['sqrt', 'log2'],
    'model__max_depth': [5, 10, None]
}
xgb_param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [3, 5],
    'model__learning_rate': [0.01, 0.1, 0.3],
    'model__subsample': [0.8, 1.0]
}

In [25]:
dt_grid = GridSearchCV(dt_pipeline, dt_param_grid, cv=5, scoring='roc_auc', n_jobs=-1, verbose=2)
rf_grid = GridSearchCV(rf_pipeline, rf_param_grid, cv=5, scoring='roc_auc', n_jobs=-1, verbose=2)
xgb_grid = GridSearchCV(xgb_pipeline, xgb_param_grid, cv=5, scoring='roc_auc', n_jobs=-1, verbose=2)

# Decision Tree Hyperparameter tuned

In [ ]:
# Decision tree tuned
dt_grid.fit(X_train, y_train)

In [ ]:
print(f"Best params: {dt_grid.best_params_}")
print(f"Best CV ROC-AUC: {dt_grid.best_score_}")

In [ ]:
dt_best = dt_grid.best_estimator_

dt_roc_auc = roc_auc_score(y_test, dt_best.predict_proba(X_test)[:, 1])
print(f"Tuned Decision tree: {dt_roc_auc:.3f}\n")
print("Classification report: \n")
print(classification_report(y_test, dt_best.predict(X_test)))

# Random forest Tuned

In [ ]:
rf_grid.fit(X_train, y_train)

In [20]:
print(f"Best params: {rf_grid.best_params_}")
print(f"Best CV ROC-AUC: {rf_grid.best_score_}")

Best params: {'model__max_depth': 10, 'model__max_features': 'sqrt', 'model__n_estimators': 300}
Best CV ROC-AUC: 0.9121091826032763


In [21]:
rf_best = rf_grid.best_estimator_

rf_roc_auc = roc_auc_score(y_test, rf_best.predict_proba(X_test)[:, 1])
print(f"Tuned Random Forest ROC-AUC: {rf_roc_auc}")
print(classification_report(y_test, rf_best.predict(X_test)))

Tuned Random Forest ROC-AUC: 0.9098213601824269
              precision    recall  f1-score   support

           0       0.87      0.95      0.91      7422
           1       0.79      0.54      0.64      2336

    accuracy                           0.85      9758
   macro avg       0.83      0.75      0.77      9758
weighted avg       0.85      0.85      0.84      9758



# XGBoost Tuned with CUDA enabled

In [ ]:
xgb_grid.fit(X_train, y_train)

In [27]:
print(f"Best params: {xgb_grid.best_params_}")
print(f"Best CV ROC-AUC: {xgb_grid.best_score_}")

Best params: {'model__learning_rate': 0.3, 'model__max_depth': 3, 'model__n_estimators': 200, 'model__subsample': 1.0}
Best CV ROC-AUC: 0.9298599464273771


In [29]:
xgb_best = xgb_grid.best_estimator_

xgb_roc_auc = roc_auc_score(y_test, xgb_best.predict_proba(X_test)[:, 1])
print("Tuned XGBoost ROC-AUC:", xgb_roc_auc)
print(classification_report(y_test, xgb_best.predict(X_test)))


Tuned XGBoost ROC-AUC: 0.9259347441704227
              precision    recall  f1-score   support

           0       0.89      0.94      0.92      7422
           1       0.78      0.64      0.70      2336

    accuracy                           0.87      9758
   macro avg       0.83      0.79      0.81      9758
weighted avg       0.87      0.87      0.87      9758

